# 🎮 Cubee — Analyse de l'entraînement Q-Learning

> **Projet IA — HENaLLux (IN252)**
> Auteurs : Victor Van Zieghem & Ethan Nickels
> Source : [VVZ-Data/Projet_IA](https://github.com/VVZ-Data/Projet_IA)

---

Ce notebook importe **directement le code source du projet** et analyse 6 agents Q-Learning
comparés par paires d'hyperparamètres.

## 🎯 Protocole — 3 paires, un paramètre isolé par paire

| Paire | Agent | α | ε₀ | γ | Paramètre isolé |
|-------|-------|---|------|---|------------------|
| **A** | IA-1 | 0.1 | **0.5** | 0.9 | ε₀ |
|       | IA-2 | 0.1 | **0.9** | 0.9 | ← défaut projet |
| **B** | IA-3 | **0.05** | 0.7 | 0.9 | α |
|       | IA-4 | **0.3** | 0.7 | 0.9 | |
| **C** | IA-5 | 0.1 | 0.7 | **0.70** | γ |
|       | IA-6 | 0.1 | 0.7 | **0.99** | |

## 📋 Sommaire
1. [Présentation du jeu](#1)
2. [Import & setup](#2)
3. [Entraînement des 6 agents](#3)
4. [Paire A — Impact de ε₀](#4)
5. [Paire B — Impact de α](#5)
6. [Paire C — Impact de γ](#6)
7. [Vue d'ensemble](#7)
8. [Conclusions par paire](#8)
9. [Sélection des paramètres — IA-7](#9)
10. [Entraînement IA-7 — Agent de production (250 000 parties)](#10)

## 1. 🎮 Présentation du jeu <a id='1'></a>

### Règles de Cubee
Deux joueurs se déplacent sur une grille **5×5**. Chaque case visitée est capturée.  
Le joueur avec le plus de cases en fin de partie gagne.

### ⚡ Mécanique centrale : `check_enclosure` (BFS — `game_model.py`)
Après chaque déplacement, un **BFS** part de la position de l'adversaire.  
Les cases vides **non atteignables** par l'adversaire sont automatiquement capturées.

```
Avant :   Après check_enclosure :
1 1 1 1   1 1 1 1
1 0 0 1   1 1 1 1   ← 0 encerclés → capturés par J1
1 0 1 1   1 1 1 1
1 1 2 2   1 1 2 2
```

### Fonctionnement de l'IA (`player.py`)

| Élément | Détail |
|---------|--------|
| Encodage état | `"{turn}_{pos2}_{pos1}_{s1}_{s2}_{board_flat}"` — plateau entier aplati |
| Récompense immédiate | `my_gain - 0.5 × opponent_gain` (cases capturées dont enclos) |
| Récompense terminale | **+10** victoire / **-10** défaite |
| Exploration (ε-greedy) | `random.choice(ALL moves)` — y compris mouvements invalides |
| Exploitation | `max Q(s,a)` sur `legal_move()` uniquement |
| Décroissance ε | multiplicative **× 0.95** jusqu'à min 0.05 |

### ⚠️ Note sur le Win Rate attendu
L'espace d'états est `3^25 ≈ 847 milliards` de combinaisons possibles.  
En 250 000 parties, chaque état est visité en moyenne **< 1 fois**.  
→ La Q-table ne converge pas en une seule session : **c'est voulu** — le projet persiste  
la Q-table dans `cubee.db` (SQLite) pour accumulation sur plusieurs sessions.  
→ Dans ce notebook, on mesure la **progression relative** entre agents, pas la performance absolue.


## 2. 📦 Import & Setup <a id='2'></a>

Nous installons les dépendances requises et importons le code source réel du jeu.
La classe `RamQTable` remplace la base de données SQLite pour éviter de polluer
`cubee.db` en production et pour accélérer significativement l’entraînement.

> ⚠️ Le notebook doit être exécuté depuis la **racine du projet** pour que `games/cubee/` soit accessible.


In [ ]:
# ── Dépendance installation ────────────────────────────────────
!pip install numpy matplotlib pandas seaborn tqdm --quiet


### Imports et configuration


In [ ]:
# ── Imports ────────────────────────────────────────────────
import random
import time
import warnings
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

# ── Import direct du code source du projet ──────────────────
from games.cubee.player import Player, AI
from games.cubee.ai_train import train_with_progress, _NB_STEPS

print('✅ Imports OK — code source réel chargé')
print(f'   _NB_STEPS  : {_NB_STEPS:.2f}  →  nb_epsilon = {int(250_000 / _NB_STEPS)} pour 250 000 parties')
print(f'   Espace d’états théorique : 3^25 = {3**25:,}')
print(f'   Couverture à 250k parties : ~{250_000 * 25 / (3**25) * 100:.6f}% de tous les états')


### Q-Table en mémoire (`RamQTable`)

Pour éviter d’écrire dans `cubee.db` pendant l’analyse, on utilise une Q-table en RAM
avec la même interface que `QTableRepo`. Cela rend l’entraînement ~10× plus rapide dans le notebook.


In [ ]:
# ── Q-Table en RAM ───────────────────────────────────────────
# Remplace SQLAlchemy/cubee.db pour le notebook — interface identique à QTableRepo

class RamQTable:
    """Q-Table en mémoire vive. Interface identique à QTableRepo."""
    def __init__(self):
        self._table = defaultdict(float)

    def get_q_value(self, gama, lr, state, action):
        return self._table[(str(gama), str(lr), state, action)]

    def update_q_value(self, gama, lr, state, action, value):
        self._table[(str(gama), str(lr), state, action)] = value

    def init_final_states(self, gama, lr): pass
    def commit(self): pass

    @property
    def size(self):
        return len(self._table)


# ── Seeds reproductibles ───────────────────────────────────────
SEEDS = {
    'IA-1': 42, 'IA-2': 42, 'IA-3': 42,
    'IA-4': 42, 'IA-5': 42, 'IA-6': 42, 'IA-7': 42,
}

# ── Palette visuelle ─────────────────────────────────────────
COLORS = {
    'IA-1': '#1a7a3c',  # vert foncé   (Paire A, ε₀=0.5)
    'IA-2': '#55c46e',  # vert vif     (Paire A, ε₀=0.9 — défaut)
    'IA-3': '#1a4fa0',  # bleu foncé   (Paire B, α=0.05)
    'IA-4': '#4da3e8',  # bleu vif     (Paire B, α=0.3)
    'IA-5': '#b94a00',  # orange foncé (Paire C, γ=0.70)
    'IA-6': '#f5a623',  # orange vif   (Paire C, γ=0.99)
    'IA-7': '#e74c3c',  # rouge        (agent optimal)
}
STYLES = {
    'IA-1': '-', 'IA-2': '--', 'IA-3': '-', 'IA-4': '--',
    'IA-5': '-', 'IA-6': '--', 'IA-7': '-',
}

print('✅ RamQTable & seeds reproductibles prêts')


## 3. 🏋️ Entraînement des 6 agents de référence (IA-1 à IA-6) <a id='3'></a>

Chaque agent joue **250 000 parties** contre un adversaire aléatoire.
Les agents s'entraînent **séquentiellement** (l'un après l'autre).

> ℹ️ **IA-7 (agent de production) n'est pas entraîné ici.** Son entraînement est déclenché
> en Section 10, après sélection des paramètres optimaux en Section 9.

Une barre `tqdm` suit la progression :
- Quel agent est en cours d'entraînement
- Combien de parties ont été jouées pour l'agent en cours

> ⚠️ **Durée estimée** : environ 25 à 45 minutes selon votre machine.
> La fenêtre de calcul est fixée à 2 500 parties (1 % du total) pour des courbes lisses.

In [ ]:
# ── Configuration des 6 agents de référence ────────────────────────────
# Chaque ligne : (nom, alpha, gamma, epsilon_initial)
# IA-7 est entraîné séparément en Section 10, après sélection des paramètres

CONFIGS = [
    ('IA-1', 0.1,  0.9,  0.5),    # Paire A — ε₀ faible
    ('IA-2', 0.1,  0.9,  0.9),    # Paire A — ε₀ élevé (défaut projet)
    ('IA-3', 0.05, 0.9,  0.7),    # Paire B — α faible
    ('IA-4', 0.3,  0.9,  0.7),    # Paire B — α élevé
    ('IA-5', 0.1,  0.70, 0.7),    # Paire C — γ faible
    ('IA-6', 0.1,  0.99, 0.7),    # Paire C — γ élevé
]

N_EPISODES = 250_000   # nombre de parties par agent
WINDOW     = 2_500     # fenêtre de calcul du win rate

PAIR_NAMES = ['IA-1', 'IA-2', 'IA-3', 'IA-4', 'IA-5', 'IA-6']  # agents de référence

print(f'Agents Section 3   : {len(CONFIGS)}  (IA-1 à IA-6)')
print(f'Parties par agent  : {N_EPISODES:,}')
print(f'Fenêtre win rate   : {WINDOW:,}')
print(f'Total parties      : {N_EPISODES * len(CONFIGS):,}')
print()
print('  ℹ️  IA-7 sera entraîné séparément en Section 10.')

### Boucle d’entraînement

La fonction `train_and_log()` encapsule `train_with_progress()` du code source du jeu.
Elle enregistre — tous les `WINDOW` parties :
- `win_rate`  : pourcentage de victoires sur la fenêtre
- `q_size`    : nombre d’entrées Q(s,a) explorées
- `epsilon`   : valeur courante du taux d’exploration


In [ ]:
# ── Fonction d'entraînement avec logs ──────────────────────────────────
def train_and_log(name: str, alpha: float, gamma: float, epsilon: float,
                  n_ep: int = N_EPISODES, window: int = WINDOW,
                  seed: int = 42, outer_bar=None) -> tuple:
    """
    Entraîne un agent Q-Learning et retourne l'agent + un DataFrame de logs.

    Colonnes du DataFrame :
      - ep         : numéro de la partie (multiple de `window`)
      - win_rate   : taux de victoire sur la fenêtre
      - wins_window: nombre de victoires dans la fenêtre
      - q_size     : nombre d'entrées Q(s,a) en RAM
      - epsilon    : valeur courante d'epsilon
    """
    random.seed(seed)
    np.random.seed(seed)

    # Création de l'agent et de son adversaire
    student  = AI(name, gama=gamma, learning_rate=alpha, epsilon=epsilon)
    student.q_table = RamQTable()
    student.init_db()
    opponent = Player('Random')

    log = defaultdict(list)
    prev_wins = prev_losses = prev_draws = 0

    # Barre de progression interne à l'agent (tqdm imbriquée)
    inner_bar = tqdm(
        total=n_ep,
        desc=f'  {name} (α={alpha}, γ={gamma}, ε₀={epsilon})',
        leave=False,
        unit='games',
    )

    def progress(current: int, total: int) -> None:
        nonlocal prev_wins, prev_losses, prev_draws

        # Mise à jour de la barre interne
        inner_bar.update(current - inner_bar.n)

        # Snapshot tous les `window` parties
        if current % window == 0 and current > 0:
            dw = student.nb_wins  - prev_wins
            dl = student.nb_loses - prev_losses
            dd = student.nb_draws - prev_draws
            wt = dw + dl + dd
            wr = dw / wt * 100 if wt else 0
            log['ep'].append(current)
            log['win_rate'].append(wr)
            log['wins_window'].append(dw)
            log['q_size'].append(student.q_table.size)
            log['epsilon'].append(student.epsilon)
            prev_wins   = student.nb_wins
            prev_losses = student.nb_loses
            prev_draws  = student.nb_draws

    train_with_progress(
        student=student, opponent=opponent,
        nb_games=n_ep, size=5,
        progress_callback=progress,
        progress_step=50,
    )

    inner_bar.close()

    # Mise à jour de la barre externe (un cran par agent terminé)
    if outer_bar is not None:
        outer_bar.update(1)

    return student, pd.DataFrame(log)

### Lancer l’entraînement

La cellule ci-dessous démarre la séquence complète d’entraînement.
Elle peut prendre **30 à 60 minutes**. Ne pas interrompre le kernel.


In [ ]:
# ── Boucle principale d'entraînement (IA-1 à IA-6) ───────────────────────
agents, logs = {}, {}

# Barre externe : indique quel agent est en cours sur les 6 agents de référence
outer_bar = tqdm(
    total=len(CONFIGS),
    desc='Progression globale',
    unit='agent',
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} agents [{elapsed}<{remaining}]'
)

for name, alpha, gamma, epsilon in CONFIGS:
    ag, df = train_and_log(name, alpha, gamma, epsilon, outer_bar=outer_bar)
    total  = ag.nb_wins + ag.nb_loses + ag.nb_draws
    wr_cum = ag.nb_wins / total * 100 if total else 0
    wr_fin = df['win_rate'].iloc[-1] if not df.empty else 0
    agents[name] = ag
    logs[name]   = df
    outer_bar.write(
        f'✅ {name:<5}  cumulative WR: {wr_cum:.1f}%'
        f'  |  final window WR: {wr_fin:.1f}%'
        f'  |  Q-table size: {ag.q_table.size:,}'
    )

outer_bar.close()
print('\n🎉 Entraînement des 6 agents de référence terminé !')

## 4. 🟢 Paire A — Impact de ε₀ <a id='4'></a>


### Rôle de ε₀

`ε₀` (epsilon initial) détermine **l’intensité de l’exploration en début d’entraînement**.
- Un ε₀ élevé : l’agent tente davantage de mouvements aléatoires → découvre plus d’états.
- Un ε₀ faible : l’agent exploite sa Q-table (encore incomplète) plus tôt.

> Les deux agents utilisent la même décroissance (×0.95) et convergent vers ε_min=0.05.
> La seule différence est **la durée passée en mode exploration**.

| | IA-1 | IA-2 |
|--|------|------|
| **α** | 0.1 | 0.1 |
| **γ** | 0.9 | 0.9 |
| **ε₀** | **0.5** | **0.9** ← défaut projet |


In [ ]:
def plot_pair(ia_a, ia_b, label_a, label_b, param_name, title):
    """3 panneaux : Win Rate (fenêtre 2 500 parties) / décroissance ε / taille Q-Table."""
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    dfa, dfb = logs[ia_a], logs[ia_b]
    ca, cb   = COLORS[ia_a], COLORS[ia_b]
    sa, sb   = STYLES[ia_a], STYLES[ia_b]

    # Win Rate sur fenêtre glissante
    ax = axs[0]
    ax.plot(dfa['ep'], dfa['win_rate'], color=ca, lw=2.5, ls=sa, label=f'{ia_a} ({label_a})')
    ax.plot(dfb['ep'], dfb['win_rate'], color=cb, lw=2.5, ls=sb, label=f'{ia_b} ({label_b})')
    ax.axhline(50, color='gray', lw=1, ls=':', alpha=0.7, label='50 % (niveau aléatoire)')
    ax.set_title('Win Rate (fenêtre de 2 500 parties)', fontweight='bold')
    ax.set_xlabel('Parties jouées'); ax.set_ylabel('Win Rate (%)')
    ax.set_ylim(0, 70); ax.legend(fontsize=9)

    # Décroissance ε
    ax = axs[1]
    ax.plot(dfa['ep'], dfa['epsilon'], color=ca, lw=2.5, ls=sa, label=f'{ia_a} ({label_a})')
    ax.plot(dfb['ep'], dfb['epsilon'], color=cb, lw=2.5, ls=sb, label=f'{ia_b} ({label_b})')
    ax.axhline(0.05, color='gray', lw=1, ls=':', alpha=0.7, label='ε_min = 0.05')
    ax.set_title('Décroissance ε (×0.95 tous les ~4 435 parties)', fontweight='bold')
    ax.set_xlabel('Parties jouées'); ax.set_ylabel('Epsilon (ε)')
    ax.set_ylim(0, 1); ax.legend(fontsize=9)

    # Taille Q-Table
    ax = axs[2]
    ax.plot(dfa['ep'], dfa['q_size'], color=ca, lw=2.5, ls=sa, label=f'{ia_a} ({label_a})')
    ax.plot(dfb['ep'], dfb['q_size'], color=cb, lw=2.5, ls=sb, label=f'{ia_b} ({label_b})')
    ax.set_title('Taille Q-Table (entrées Q(s,a))', fontweight='bold')
    ax.set_xlabel('Parties jouées'); ax.set_ylabel('Entrées')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(f'cubee_pair_{param_name}.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_pair('IA-1', 'IA-2',
          'ε₀=0.5 — exploration faible',
          'ε₀=0.9 — exploration élevée (défaut)',
          'A_epsilon',
          '🟢 Paire A — Impact de ε₀  |  α=0.1, γ=0.9  |  seul ε₀ varie')


In [ ]:
# ── Analyse statistique Paire A ─────────────────────────────────
wr1_early = logs['IA-1']['win_rate'].iloc[:3].mean()
wr2_early = logs['IA-2']['win_rate'].iloc[:3].mean()
wr1_late  = logs['IA-1']['win_rate'].iloc[-3:].mean()
wr2_late  = logs['IA-2']['win_rate'].iloc[-3:].mean()
q1, q2    = agents['IA-1'].q_table.size, agents['IA-2'].q_table.size

print('📊 Paire A — ε₀')
print('─' * 60)
print(f'  Début (3 premières fenêtres)  →  IA-1 : {wr1_early:.1f}%  |  IA-2 : {wr2_early:.1f}%')
print(f'  Fin   (3 dernières fenêtres)  →  IA-1 : {wr1_late:.1f}%  |  IA-2 : {wr2_late:.1f}%')
print(f'  Taille Q-table finale       →  IA-1 : {q1:,}  |  IA-2 : {q2:,}')
print()
print('  Analyse :')
print('  → ε₀ plus élevé = plus d’états explorés → Q-table plus grande.')
print('  → Les deux convergent vers ε=0.05 (même décroissance ×0.95).')
print('  → ε₀=0.5 exploite plus tôt mais avec une Q-table moins complète.')
better_early = 'IA-1 (ε₀=0.5)' if wr1_early > wr2_early else 'IA-2 (ε₀=0.9)'
better_late  = 'IA-1 (ε₀=0.5)' if wr1_late  > wr2_late  else 'IA-2 (ε₀=0.9)'
print(f'  → Meilleur en début : {better_early}.')
print(f'  → Meilleur en fin   : {better_late}.')


### 📌 Paire A — Conclusion

L'écart entre IA-1 et IA-2 est **faible** (1–2 %).
Cela suggère que ε₀ a un **impact limité** une fois que l'entraînement est assez long
pour que les deux agents convergent vers ε_min=0.05.

Cependant, IA-1 (ε₀=0.5) tend à mieux performer **dans les dernières fenêtres**
parce qu'il commence à exploiter sa Q-table plus tôt.

**Recommandation** : → Voir Section 9 pour la sélection finale du paramètre ε₀.

## 5. 🔵 Paire B — Impact de α <a id='5'></a>


### Rôle de α

`α` (taux d’apprentissage) détermine **l’intensité de la mise à jour de la Q-table**
après chaque expérience.
- Un α faible (0.05) intègre chaque expérience doucement → apprentissage lent mais stable.
- Un α élevé (0.30) intègre chaque expérience agressivement → plus rapide mais plus bruyté.

> Dans Cubee, l’espace d’états est ~847 milliards. La plupart des états sont visités
> **au plus une fois** en 250 000 parties. Un α élevé sur des états rarement revisités
> peut écraser des valeurs Q valides avec le bruit d’une seule partie.

| | IA-3 | IA-4 |
|--|------|------|
| **α** | **0.05** | **0.30** |
| **γ** | 0.9 | 0.9 |
| **ε₀** | 0.7 | 0.7 |


In [ ]:
plot_pair('IA-3', 'IA-4',
          'α=0.05 — mise à jour lente',
          'α=0.30 — mise à jour rapide',
          'B_alpha',
          '🔵 Paire B — Impact de α  |  ε₀=0.7, γ=0.9  |  seul α varie')


In [ ]:
# ── Analyse statistique Paire B ─────────────────────────────────
wr3_early = logs['IA-3']['win_rate'].iloc[:3].mean()
wr4_early = logs['IA-4']['win_rate'].iloc[:3].mean()
wr3_late  = logs['IA-3']['win_rate'].iloc[-3:].mean()
wr4_late  = logs['IA-4']['win_rate'].iloc[-3:].mean()
half = len(logs['IA-3']) // 2
std3 = logs['IA-3']['win_rate'].iloc[half:].std()
std4 = logs['IA-4']['win_rate'].iloc[half:].std()

print('📊 Paire B — α')
print('─' * 60)
print(f'  Début (3 premières fenêtres)  →  IA-3 : {wr3_early:.1f}%  |  IA-4 : {wr4_early:.1f}%')
print(f'  Fin   (3 dernières fenêtres)  →  IA-3 : {wr3_late:.1f}%  |  IA-4 : {wr4_late:.1f}%')
print(f'  Stabilité (std 2ème moitié)  →  IA-3 : {std3:.2f}  |  IA-4 : {std4:.2f}')
print(f'  Taille Q-table finale       →  IA-3 : {agents["IA-3"].q_table.size:,}  |  IA-4 : {agents["IA-4"].q_table.size:,}')
print()
print('  Analyse :')
print('  → α=0.05 : chaque expérience modifie peu la Q-table → apprentissage lent mais stable.')
print('  → α=0.30 : mises à jour agressives → plus rapide mais potentiellement plus d’oscillations.')
more_stable = 'IA-3 (α=0.05)' if std3 <= std4 else 'IA-4 (α=0.30)'
print(f'  → {more_stable} est plus stable sur la 2ème moitié de l’entraînement.')


### 📌 Paire B — Conclusion

L’écart entre IA-3 et IA-4 est **significatif**.
IA-3 (α=0.05) surpasse constamment IA-4 (α=0.30) dans les dernières fenêtres.

**Pourquoi ?** Dans un espace d’états très grand où les états sont rarement revisités,
un α élevé **écrase** les valeurs Q avec le résultat d’une seule partie.
Un α faible moyenne sur plusieurs visites et produit des valeurs Q plus fiables.

**Recommandation** : α=0.05 est clairement préférable pour Cubee.


## 6. 🟠 Paire C — Impact de γ <a id='6'></a>


### Rôle de γ

`γ` (facteur de discount) détermine **la valeur accordée aux récompenses futures**.
- γ=0.70 → l’agent favorise les **captures immédiates** de cases.
- γ=0.99 → l’agent propage la récompense terminale (+10 victoire / −10 défaite)
  loin en arrière dans le temps, apprenant à **planifier des enclos** sur plusieurs coups.

> Dans Cubee, le signal de récompense principal est terminal (+10/−10). Un γ élevé
> permet aux valeurs Q de « se souvenir » qu’une séquence de coups mène à une victoire,
> même si aucune capture immédiate ne s’est produite entre-temps.
> C’est pourquoi γ=0.9 a été choisi comme défaut dans le projet.

| | IA-5 | IA-6 |
|--|------|------|
| **α** | 0.1 | 0.1 |
| **γ** | **0.70** | **0.99** |
| **ε₀** | 0.7 | 0.7 |


In [ ]:
plot_pair('IA-5', 'IA-6',
          'γ=0.70 — court terme',
          'γ=0.99 — long terme',
          'C_gamma',
          '🟠 Paire C — Impact de γ  |  α=0.1, ε₀=0.7  |  seul γ varie')


In [ ]:
# ── Analyse statistique Paire C ─────────────────────────────────
wr5_early = logs['IA-5']['win_rate'].iloc[:3].mean()
wr6_early = logs['IA-6']['win_rate'].iloc[:3].mean()
wr5_late  = logs['IA-5']['win_rate'].iloc[-3:].mean()
wr6_late  = logs['IA-6']['win_rate'].iloc[-3:].mean()

print('📊 Paire C — γ')
print('─' * 60)
print(f'  Début (3 premières fenêtres)  →  IA-5 : {wr5_early:.1f}%  |  IA-6 : {wr6_early:.1f}%')
print(f'  Fin   (3 dernières fenêtres)  →  IA-5 : {wr5_late:.1f}%  |  IA-6 : {wr6_late:.1f}%')
print(f'  Taille Q-table finale       →  IA-5 : {agents["IA-5"].q_table.size:,}  |  IA-6 : {agents["IA-6"].q_table.size:,}')
print()
print('  Analyse :')
print('  → Signal principal : +10 victoire / −10 défaite (terminal).')
print('  → γ=0.70 : actualise fortement le futur → optimise les captures immédiates.')
print('  → γ=0.99 : propage la récompense terminale loin en arrière → apprend à planifier')
print('    des enclos sur plusieurs coups (mécanique check_enclosure BFS).')
better = 'IA-6 (γ=0.99)' if wr6_late >= wr5_late else 'IA-5 (γ=0.70)'
print(f'  → {better} performe mieux en fin d’entraînement.')
print(f'    Cohérent avec γ=0.9 choisi comme défaut dans le projet.')


### 📌 Paire C — Conclusion

L’écart entre IA-5 et IA-6 est **le plus grand des trois paires**.
IA-6 (γ=0.99) surpasse largement IA-5 (γ=0.70) dans les dernières fenêtres.

**Pourquoi ?** Dans Cubee, la récompense terminale (+10/−10) est le signal le plus important.
Un γ élevé assure que ce signal se propage en arrière sur toute la trajectoire de jeu,
permettant à l’agent d’apprendre que des décisions positionnelles en début de partie
mènent à la victoire. Un γ faible « oublie » la récompense terminale trop rapidement.

**Recommandation** : γ=0.99 (ou proche) est l’amélioration la plus impactante possible.
C’est **l’hyperparamètre le plus important** pour ce jeu.


## 7. 📊 Vue d’ensemble — 6 agents de référence <a id='7'></a>

Cette section compare les 6 agents du protocole côte à côte.


In [ ]:
# ── Vue d'ensemble des 6 agents de référence (IA-1 à IA-6) ─────────────
PAIR_CONFIGS = [c for c in CONFIGS if c[0] != 'IA-7']

fig = plt.figure(figsize=(18, 13))
fig.suptitle(
    '📊 Comparaison globale — 6 agents Q-Learning · Cubee\n'
    'Code source réel · grille 5×5 · check_enclosure BFS · ε ×0.95 · fenêtre WR 2 500',
    fontsize=13, fontweight='bold', y=1.01)

gs = fig.add_gridspec(3, 3, hspace=0.50, wspace=0.35)

# ── Ligne 1 : Win Rate par paire ──────────────────────────────────
pairs = [
    ('IA-1', 'IA-2', 'Paire A — ε₀\n(α=0.1, γ=0.9)'),
    ('IA-3', 'IA-4', 'Paire B — α\n(ε₀=0.7, γ=0.9)'),
    ('IA-5', 'IA-6', 'Paire C — γ\n(α=0.1, ε₀=0.7)'),
]
for col, (a, b, ttl) in enumerate(pairs):
    ax = fig.add_subplot(gs[0, col])
    ax.plot(logs[a]['ep'], logs[a]['win_rate'], color=COLORS[a], lw=2.2, ls=STYLES[a], label=a)
    ax.plot(logs[b]['ep'], logs[b]['win_rate'], color=COLORS[b], lw=2.2, ls=STYLES[b], label=b)
    ax.axhline(50, color='gray', lw=1, ls=':', alpha=0.6)
    ax.set_title(f'Win Rate\n{ttl}', fontsize=10, fontweight='bold')
    ax.set_ylim(0, 70); ax.set_xlabel('Parties'); ax.set_ylabel('Win Rate (%)')
    ax.legend(fontsize=8)

# ── Ligne 2 : toutes les 6 superposées ────────────────────────────────
ax_all = fig.add_subplot(gs[1, :])
for name, alpha, gamma, epsilon in PAIR_CONFIGS:
    ax_all.plot(logs[name]['ep'], logs[name]['win_rate'],
                color=COLORS[name], lw=2.2, ls=STYLES[name],
                label=f'{name}  α={alpha}, γ={gamma}, ε₀={epsilon}')
ax_all.axhline(50, color='gray', lw=1.5, ls='--', alpha=0.5, label='50 % (niveau aléatoire)')
ax_all.set_title('6 agents — Win Rate superposé (fenêtre de 2 500 parties)', fontsize=12, fontweight='bold')
ax_all.set_ylim(0, 70); ax_all.set_xlabel('Parties'); ax_all.set_ylabel('Win Rate (%)')
ax_all.legend(fontsize=8, loc='upper right', ncol=2)

# ── Ligne 3 : WR fenêtre finale + taille Q-Table ──────────────────────
names_list = [n for n, *_ in PAIR_CONFIGS]
wr_finals  = [logs[n]['win_rate'].iloc[-1] for n in names_list]
q_finals   = [agents[n].q_table.size for n in names_list]
bar_colors = [COLORS[n] for n in names_list]

ax_bar = fig.add_subplot(gs[2, :2])
bars = ax_bar.bar(names_list, wr_finals, color=bar_colors, edgecolor='white', linewidth=1.5, width=0.6)
ax_bar.axhline(50, color='gray', lw=1.5, ls='--', alpha=0.7)
for bar, wr in zip(bars, wr_finals):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{wr:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
for x, (n, *_) in enumerate(PAIR_CONFIGS):
    p = 'A' if n in ('IA-1', 'IA-2') else ('B' if n in ('IA-3', 'IA-4') else 'C')
    ax_bar.text(x, 1, f'Paire {p}', ha='center', fontsize=8, color='white', fontweight='bold')
ax_bar.set_title('Win Rate — dernière fenêtre de 2 500 parties', fontsize=11, fontweight='bold')
ax_bar.set_ylabel('Win Rate (%)'); ax_bar.set_ylim(0, 75)

ax_q = fig.add_subplot(gs[2, 2])
ax_q.barh(names_list, q_finals, color=bar_colors, edgecolor='white', linewidth=1.5)
for i, q in enumerate(q_finals):
    ax_q.text(q + max(q_finals) * 0.01, i, f'{q:,}', va='center', fontsize=8)
ax_q.set_title('Taille Q-Table finale\n(entrées Q(s,a) en RAM)', fontsize=11, fontweight='bold')
ax_q.set_xlabel('Entrées Q(s,a)')

plt.savefig('cubee_global_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figure sauvegardée : cubee_global_comparison.png')


### Lecture des résultats

Le diagramme en barres montre le **win rate sur la dernière fenêtre de 2 500 parties**
pour chaque agent. C’est plus informatif que le win rate cumulé, qui inclut la phase
aléatoire du début.

**Observations clés :**
- **γ a le plus grand impact** (écart Paire C : ~10 points)
- **α a un impact significatif** (écart Paire B : ~8 points)
- **ε₀ a un impact limité** (écart Paire A : ~1–2 points)

Ces résultats vont guider la construction de l’agent optimal présenté en section 8.


## 8. 📝 Conclusions par paire <a id='8'></a>

Cette section synthétise les résultats des 3 paires d'hyperparamètres.

In [ ]:
# ── Tableau de synthèse — agents de référence (IA-1 à IA-6) ─────────────
print('=' * 70)
print('      📊 SYNTHÈSE — 6 AGENTS Q-LEARNING · CUBEE')
print('=' * 70)
print(f'  {"Nom":<6}  {"α":>5}  {"γ":>5}  {"ε₀":>5}  '
      f'{"V/Total":>12}  {"WR final":>10}  {"Q-Table":>12}  Paire')
print('  ' + '─' * 68)
for name, alpha, gamma, epsilon in CONFIGS:
    if name == 'IA-7':
        continue
    ag    = agents[name]
    total = ag.nb_wins + ag.nb_loses + ag.nb_draws
    wr_f  = logs[name]['win_rate'].iloc[-1]
    p     = 'A (ε₀)' if name in ('IA-1', 'IA-2') else \
            'B (α)'  if name in ('IA-3', 'IA-4') else 'C (γ)'
    print(f'  {name:<6}  {alpha:>5}  {gamma:>5}  {epsilon:>5}  '
          f'  {ag.nb_wins:>5}/{total:<5}  {wr_f:>8.1f}%  {ag.q_table.size:>12,}  {p}')

wr_v = {n: logs[n]['win_rate'].iloc[-1] for n in PAIR_NAMES}
best = max(wr_v, key=wr_v.get)
print()
print('  ┌─────────────────────────────────────────────────────────────────┐')
print('  │  POINTS CLÉS                                                     │')
print('  ├─────────────────────────────────────────────────────────────────┤')
print(f'  │  🏆 Meilleur agent par paire (fenêtre finale) : {best} ({wr_v[best]:.1f}%)      │')
print(f'  │                                                                 │')
print(f'  │  Paire A (ε₀) : IA-1={wr_v["IA-1"]:.1f}% vs IA-2={wr_v["IA-2"]:.1f}%              │')
print(f'  │  Paire B (α)  : IA-3={wr_v["IA-3"]:.1f}% vs IA-4={wr_v["IA-4"]:.1f}%              │')
print(f'  │  Paire C (γ)  : IA-5={wr_v["IA-5"]:.1f}% vs IA-6={wr_v["IA-6"]:.1f}%              │')
print(f'  │                                                                 │')
print(f'  │  ⚠️  250 000 parties = ~0.001% de l\'espace 3^25 d\'états.          │')
print(f'  │      La comparaison RELATIVE entre paires reste valide.         │')
print(f'  │                                                                 │')
print(f'  │  Défauts projet : α=0.1, γ=0.9, ε₀=0.9 (= IA-2)               │')
print('  └─────────────────────────────────────────────────────────────────┘')

### ⚠️ Nuance sur la Paire A (ε₀)

Le tableau ci-dessus indique IA-1 (ε₀=0.5) à 47.7% et IA-2 (ε₀=0.9) à 46.7%, soit un écart
de **1 point de pourcentage** en faveur de IA-1.

Cet écart est **non significatif** sur 10 000 parties, pour deux raisons :
- La variance est élevée (std ≈ 4-5pp) : un écart de 1pp est dans le bruit statistique.
- Sur peu de parties, ε₀=0.5 exploite plus tôt → meilleur score court terme, mais avec une
  Q-table moins complète (7 286 676 entrées vs 9 356 062 pour IA-2, soit **28% d'états
  supplémentaires** explorés par IA-2).

Sur 250 000 parties, l'exploration maximale (ε₀=0.9) est indispensable pour couvrir
suffisamment l'espace d'états (3^25 ≈ 847 milliards de combinaisons possibles).

→ **Le paramètre retenu pour IA-7 est ε₀=0.9.** La justification complète est en Section 9.

In [ ]:
# ── Recommandations finales ────────────────────────────────────────────
print('=' * 70)
print('  RECOMMANDATIONS FINALES — CUBEE Q-LEARNING')
print('=' * 70)
print()
print('  CLASSEMENT DES PARAMÈTRES PAR IMPACT (fort → faible) :')
print('  1. γ (facteur de discount) — IMPACT LE PLUS FORT')
print('     → Utiliser γ=0.99. La récompense terminale (+10/−10) doit se propager loin.')
print('     → γ=0.9 (défaut actuel) est acceptable mais sous-optimal.')
print()
print('  2. α (taux d\'apprentissage) — IMPACT SIGNIFICATIF')
print('     → Utiliser α=0.05. Avec 847 milliards d\'états, la plupart ne sont visités qu\'une fois.')
print('     → Un α faible empêche une expérience bruyante de corrompre les valeurs Q.')
print('     → α=0.1 (défaut actuel) est légèrement trop élevé.')
print()
print('  3. ε₀ (epsilon initial) — IMPACT LIMITÉ')
print('     → Voir Section 9 pour la sélection finale du paramètre ε₀.')
print()
print('  CONFIGURATION RECOMMANDÉE POUR LE PROJET :')
print('  ┌─────────────────────────────────────────────────────┐')
print('  │  α = 0.05   γ = 0.99   ε₀ = 0.9   (= IA-7)       │')
print('  └─────────────────────────────────────────────────────┘')
print()
print('  LIMITES DE CETTE ANALYSE :')
print('  ⚠️  250 000 parties couvrent < 0.001% de l\'espace d\'états.')
print('     Les résultats montrent des tendances d\'apprentissage, pas de convergence absolue.')
print('  ⚠️  Une seule seed utilisée. Les résultats peuvent varier.')
print('  ⚠️  Seulement 2 valeurs testées par paramètre (pas de grid search complet).')
print('  ⚠️  Les interactions entre paramètres ne sont pas testées (IA-7 = une seule combinaison).')

### Perspectives

Pour aller plus loin dans l'analyse :
- **Utiliser plusieurs seeds** pour mesurer la variance et confirmer la significativité statistique.
- **Augmenter le budget d'entraînement** — 1 000 000+ parties donneraient des courbes de
  convergence beaucoup plus nettes.
- **Grid search** — tester toutes les combinaisons de
  α ∈ {0.01, 0.05, 0.1} × γ ∈ {0.9, 0.95, 0.99}.
- **Self-play** — entraîner IA contre IA (et non contre Random) pour accélérer
  la couverture de la Q-table.
- **Amélioration de l'encodage d'état** — l'encodage actuel (plateau entier aplati,
  3^25 états) est trop grand. Un encodage local compact (fenêtre 5×5 centrée sur l'agent)
  pourrait réduire drastiquement l'espace d'états et permettre une vraie convergence
  en un temps raisonnable.

## 9. 🏆 Sélection des paramètres — IA-7 <a id='9'></a>

Sur la base de l'analyse par paires (Sections 4, 5, 6), on sélectionne le meilleur paramètre
de chaque paire pour construire l'agent de production.

| Paire | Paramètre | Valeur retenue | Justification |
|-------|-----------|----------------|---------------|
| **A** | ε₀ | **0.9** | Écart de 1pp non significatif sur 10k parties (dans la variance). IA-2 explore 28% d'états de plus que IA-1. Sur 250k parties, la couverture de l'espace d'états est déterminante. |
| **B** | α | **0.05** | Écart de +9.0pp (53.7% vs 44.7%), statistiquement solide. Apprentissage lent mais stable ; réduit les oscillations sur longues durées. |
| **C** | γ | **0.99** | Écart de +10.6pp (56.1% vs 45.5%), le plus large de toutes les paires. γ proche de 1 propage la récompense terminale (+10 victoire) sur de nombreux coups en arrière → meilleure planification des enclos (`check_enclosure`). |

**Paramètres finaux retenus pour IA-7 :**

| Paramètre | Valeur |
|-----------|--------|
| α (learning rate) | **0.05** |
| γ (discount factor) | **0.99** |
| ε₀ (epsilon initial) | **0.9** |
| Décroissance ε | ×0.95 par épisode, min 0.05 (inchangé) |
| Nb parties | **250 000** (vs 10 000 pour IA-1 à IA-6) |
| Adversaire | Random (inchangé) |

In [ ]:
# Validation numérique de la sélection des paramètres
# Affiche les Win Rates finaux des gagnants de chaque paire

print('=' * 60)
print('  🏆 SÉLECTION DES PARAMÈTRES — IA-7')
print('=' * 60)

pairs = [
    ('A', 'ε₀', 'IA-1', 'ε₀=0.5', 'IA-2', 'ε₀=0.9 ✅'),
    ('B', 'α',  'IA-3', 'α=0.05 ✅', 'IA-4', 'α=0.30'),
    ('C', 'γ',  'IA-5', 'γ=0.70', 'IA-6', 'γ=0.99 ✅'),
]

for pair_name, param, ia_a, label_a, ia_b, label_b in pairs:
    wr_a = logs[ia_a]['win_rate'].iloc[-1]
    wr_b = logs[ia_b]['win_rate'].iloc[-1]
    delta = abs(wr_b - wr_a)
    winner_label = label_a if wr_a >= wr_b else label_b
    print(f'  Paire {pair_name} ({param}) : {ia_a} ({label_a}) = {wr_a:.1f}%'
          f'  |  {ia_b} ({label_b}) = {wr_b:.1f}%'
          f'  →  Δ={delta:.1f}pp')

print()
print('  Paramètres retenus : α=0.05 | γ=0.99 | ε₀=0.9')
print('  Durée entraînement : 250 000 parties')
print('=' * 60)

## 10. 🤖 Entraînement IA-7 — Agent de production (250 000 parties) <a id='10'></a>

IA-7 est entraîné avec les paramètres sélectionnés en Section 9 : **α=0.05, γ=0.99, ε₀=0.9**.

Contrairement aux agents IA-1 à IA-6 (10 000 parties), cet entraînement utilise **250 000
parties** pour permettre une meilleure convergence malgré l'espace d'états gigantesque
(3^25 ≈ 847 milliards de combinaisons).

> **Note sur la comparabilité** : les courbes de IA-7 ne sont pas directement comparables
> à celles de IA-1 à IA-6 sur le même axe (échelle ×25). Les cellules ci-dessous affichent
> IA-7 sur sa propre échelle, puis une comparaison équitable sur une fenêtre de 10 000
> parties en fin d'entraînement.

In [ ]:
# ── Paramètres de IA-7 (agent de production) ──────────────────────────
ALPHA_7    = 0.05
GAMMA_7    = 0.99
EPSILON_7  = 0.9    # ε₀=0.9 retenu — voir Section 9
NB_GAMES_7 = 250_000

print('=' * 60)
print('  🚀 ENTRAÎNEMENT DE IA-7 — AGENT DE PRODUCTION')
print('=' * 60)
print(f'  α (learning rate)   : {ALPHA_7}')
print(f'  γ (discount factor) : {GAMMA_7}')
print(f'  ε₀ (epsilon initial): {EPSILON_7}')
print(f'  Parties             : {NB_GAMES_7:,}')
print()
print('  ⚠️  Durée estimée : 5 à 15 minutes selon votre machine.')
print('=' * 60)

ia7_outer = tqdm(
    total=1,
    desc='IA-7 — Agent de production',
    unit='agent',
    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} agent [{elapsed}<{remaining}]'
)

agents['IA-7'], logs['IA-7'] = train_and_log(
    name='IA-7',
    alpha=ALPHA_7,
    gamma=GAMMA_7,
    epsilon=EPSILON_7,
    n_ep=NB_GAMES_7,
    seed=SEEDS.get('IA-7', 42),
    outer_bar=ia7_outer,
)

ia7_outer.close()

_ag7   = agents['IA-7']
_total = _ag7.nb_wins + _ag7.nb_loses + _ag7.nb_draws
_wr_c  = _ag7.nb_wins / _total * 100 if _total else 0
_wr_f  = logs['IA-7']['win_rate'].iloc[-1]

print(f'\n✅ IA-7 entraîné !')
print(f'   WR cumulé           : {_wr_c:.1f}%')
print(f'   WR dernière fenêtre : {_wr_f:.1f}%')
print(f'   Q-table             : {_ag7.q_table.size:,} entrées')

In [ ]:
# ── Comparaison IA-7 vs meilleurs agents de chaque paire ────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(
    '🤖 IA-7 (Agent de production) vs Meilleur agent par paire\n'
    '250 000 parties · fenêtre = 2 500',
    fontsize=13, fontweight='bold'
)

# --- Graphique gauche : courbes win rate ---
ax = axes[0]
comparisons = [
    ('IA-1', 'Meilleur Paire A (ε₀=0.5)',  '#2ecc71', '--'),
    ('IA-3', 'Meilleur Paire B (α=0.05)',  '#3498db', '--'),
    ('IA-6', 'Meilleur Paire C (γ=0.99)',  '#e67e22', '--'),
    ('IA-7', 'IA-7 Agent de production',    '#e74c3c', '-'),
]
for name, label, color, ls in comparisons:
    ax.plot(logs[name]['ep'], logs[name]['win_rate'],
            color=color, lw=2.5, ls=ls, label=label)
ax.axhline(50, color='gray', lw=1, ls=':', alpha=0.6, label='50 % (niveau aléatoire)')
ax.set_title('Évolution du Win Rate', fontsize=11, fontweight='bold')
ax.set_ylim(0, 75)
ax.set_xlabel('Parties jouées')
ax.set_ylabel('Win Rate (%)')
ax.legend(fontsize=9)

# --- Graphique droit : barres win rate final ---
ax2 = axes[1]
names_cmp  = ['IA-1', 'IA-3', 'IA-6', 'IA-7']
labels_cmp = ['Meilleur Paire A\n(ε₀=0.5)', 'Meilleur Paire B\n(α=0.05)',
              'Meilleur Paire C\n(γ=0.99)', 'IA-7\nAgent de production']
colors_cmp = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']
wr_finals  = [logs[n]['win_rate'].iloc[-1] for n in names_cmp]

bars = ax2.bar(labels_cmp, wr_finals, color=colors_cmp,
               edgecolor='white', linewidth=1.5, width=0.55)
ax2.axhline(50, color='gray', lw=1.5, ls='--', alpha=0.7)
for bar, wr in zip(bars, wr_finals):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f'{wr:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_title('Win Rate — dernière fenêtre', fontsize=11, fontweight='bold')
ax2.set_ylabel('Win Rate (%)')
ax2.set_ylim(0, 80)

plt.tight_layout()
plt.savefig('cubee_ia7_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figure sauvegardée : cubee_ia7_comparison.png')

In [ ]:
# ── Statistiques détaillées de IA-7 ───────────────────────────────
ag7   = agents['IA-7']
total = ag7.nb_wins + ag7.nb_loses + ag7.nb_draws
wr_cum = ag7.nb_wins / total * 100 if total else 0
wr_fin = logs['IA-7']['win_rate'].iloc[-1]

print('=' * 60)
print('  IA-7 — AGENT DE PRODUCTION — RÉSULTATS')
print('=' * 60)
print(f'  Paramètres    : α=0.05, γ=0.99, ε₀=0.9')
print(f'  Parties      : {total:,}')
print(f'  WR cumulé    : {wr_cum:.1f}%')
print(f'  WR final     : {wr_fin:.1f}%')
print(f'  Q-table      : {ag7.q_table.size:,} entrées')
print()

best_per_pair = [('IA-1', 'Paire A'), ('IA-3', 'Paire B'), ('IA-6', 'Paire C')]
print('  Comparaison vs meilleur agent par paire (WR fenêtre finale) :')
print('  ' + '─' * 42)
for comp_name, pair_label in best_per_pair:
    wr_comp = logs[comp_name]['win_rate'].iloc[-1]
    delta   = wr_fin - wr_comp
    sign    = '+' if delta >= 0 else ''
    print(f'  vs {comp_name} ({pair_label:<8}) : {wr_comp:.1f}% → IA-7 {sign}{delta:.1f} pts')

### Interprétation

Si IA-7 surpasse **les trois gagnants de paire**, cela valide l'hypothèse d'indépendance :
les trois hyperparamètres peuvent être optimisés séparément et combinés additivement.

Si IA-7 **ne surpasse pas** les gagnants, cela suggère des interactions entre paramètres —
par exemple, un α très faible combiné à un γ très élevé peut nécessiter davantage de parties
pour converger, car les valeurs Q se mettent à jour plus lentement et la récompense
à long horizon se propage plus progressivement.

Ce résultat oriente la **recommandation de configuration** pour le projet (voir section 9).